In [1]:
# uv add ipykernel
import faiss
import numpy as np

In [2]:
# 步骤1：准备数据
items = [
    {"type": "phone", "id": "用户A", "number": 13800001234},
    {"type": "phone", "id": "用户B", "number": 13800005678},
    {"type": "order", "id": "订单1001", "number": 203011010001},
    {"type": "order", "id": "订单1002", "number": 203011010123},
    {"type": "order", "id": "订单2001", "number": 203012150045},
    {"type": "phone", "id": "用户C", "number": 13912345678},
    {"type": "phone", "id": "用户D", "number": 13798765432},
    {"type": "order", "id": "订单3001", "number": 205001020333},
    {"type": "order", "id": "订单3002", "number": 205001020777},
    {"type": "phone", "id": "用户E", "number": 13622223333},
]
# FAISS数据不支持int64,所以需要转换成float32
numbers = np.array([row['number'] for row in items ],dtype='float32')
numbers

array([1.3800002e+10, 1.3800006e+10, 2.0301101e+11, 2.0301101e+11,
       2.0301215e+11, 1.3912346e+10, 1.3798766e+10, 2.0500102e+11,
       2.0500102e+11, 1.3622223e+10], dtype=float32)

In [3]:
# 步骤2: 构建数据_放到数据
# 建立索引
dimension = 1
index = faiss.IndexFlatL2(dimension)
# 添加数据
vectors = numbers.reshape(-1,1)
index.add(vectors)
print(f'已向索引添加{len(items)}个数字向量(维度={dimension})')

已向索引添加10个数字向量(维度=1)


In [6]:
# 步骤3：检索数据
query_number = 205001020500
query_vec = np.array([[query_number]],dtype='float32')

# 获取相近多个数据
k = 5
distances, indices =index.search(query_vec,k)  # distances 表示相似度的距离, indices索引的位置
# 取[0]是为了降维，方便获取里面的数据
distances, indices = distances[0], indices[0] 
distances, indices

(array([0.0000000e+00, 0.0000000e+00, 3.9556044e+18, 3.9601025e+18,
        3.9601025e+18], dtype=float32),
 array([7, 8, 4, 2, 3]))

In [8]:
distances, indices

(array([0.0000000e+00, 0.0000000e+00, 3.9556044e+18, 3.9601025e+18,
        3.9601025e+18], dtype=float32),
 array([7, 8, 4, 2, 3]))

In [11]:
print(f'查询数字:{query_number}')
print(f'Top-{k} 最相近的数字(L2距离，越小越相近)')
for rank, (i,d) in enumerate(zip(indices,distances),start =1):
    row = items[int(i)]
    print(f'{rank} 距离{d:.0f} | 类型:{row["type"]}   | 标识{row["id"]}  | 数字：{row["number"]}')

查询数字:205001020500
Top-5 最相近的数字(L2距离，越小越相近)
1 距离0 | 类型:order   | 标识订单3001  | 数字：205001020333
2 距离0 | 类型:order   | 标识订单3002  | 数字：205001020777
3 距离3955604406476472320 | 类型:order   | 标识订单2001  | 数字：203012150045
4 距离3960102508545703936 | 类型:order   | 标识订单1001  | 数字：203011010001
5 距离3960102508545703936 | 类型:order   | 标识订单1002  | 数字：203011010123
